# Retrieval and Generation

In [7]:
import os
import pickle
from dotenv import load_dotenv

from langchain.vectorstores import FAISS
from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_openai import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

In [2]:
load_dotenv()

True

In [6]:
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

DATA_PATH = "../data"
FAISS_INDEX_PATH = os.path.join(DATA_PATH, "faiss_index")
DOCS_PATH = os.path.join(FAISS_INDEX_PATH, "docs.pkl")

EMBEDDINGS = OpenAIEmbeddings()
LLM = ChatOpenAI(model = "gpt-4o-mini", temperature = 0)

In [5]:
vectorstore = FAISS.load_local(
    folder_path = FAISS_INDEX_PATH, 
    embeddings = EMBEDDINGS,
    allow_dangerous_deserialization = True
)

In [8]:
if os.path.exists(DOCS_PATH):
    with open(DOCS_PATH, "rb") as f:
        docs = pickle.load(f)

    texts = [d.page_content for d in docs]
else:
    texts = [v.page_content for v in vectorstore.docstore._dict.values()]

In [11]:
bm5_retriever = BM25Retriever.from_texts(texts)
bm5_retriever.k = 3

In [12]:
vector_retriever = vectorstore.as_retriever(search_kwargs = {"k": 3})

In [13]:
hybrid_retrieval = EnsembleRetriever(
    retrievers = [bm5_retriever, vector_retriever],
    weights = [0.4, 0.6]
)

In [14]:
qa_chain = RetrievalQA.from_chain_type(
    llm = LLM,
    retriever = hybrid_retrieval,
    return_source_documents = True
)

In [15]:
test_query = "Is there any anomalies on the machines?"

In [16]:
response = qa_chain.invoke({"query": test_query})

In [17]:
response["result"]

'Yes, there are anomalies detected on the machines. The CX-450 has a spindle temperature anomaly detected on 02/03/2025, and the HPX-1200 has anomalies for oil temperature and pressure, both detected on the same date.'